In [8]:
import pandas as pd
import numpy as np
import sklearn
import xgboost
import shap

print("Healthcare Analytics Environment Ready!")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)
print("SHAP:", shap.__version__)

Healthcare Analytics Environment Ready!
Pandas: 3.0.5
NumPy: 2.5.2
Scikit-learn: 1.9.0
XGBoost: 3.4.1
SHAP: 0.52.0


In [24]:
from ucimlrepo import fetch_ucirepo

# Download dataset from UCI
dataset = fetch_ucirepo(id=296)

# Features and target
X = dataset.data.features
y = dataset.data.targets

print("Dataset downloaded successfully!")
print("Features shape:", X.shape)
print("Target shape:", y.shape)

Dataset downloaded successfully!
Features shape: (101766, 47)
Target shape: (101766, 1)


/Users/aniketpandey/healthcare-readmission-analytics/.venv/lib/python3.12/site-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


In [25]:
# Basic dataset information

print("Number of rows:", X.shape[0])
print("Number of columns:", X.shape[1])

print("\nColumn names:")
for i, col in enumerate(X.columns, start=1):
    print(f"{i}. {col}")

Number of rows: 101766
Number of columns: 47

Column names:
1. race
2. gender
3. age
4. weight
5. admission_type_id
6. discharge_disposition_id
7. admission_source_id
8. time_in_hospital
9. payer_code
10. medical_specialty
11. num_lab_procedures
12. num_procedures
13. num_medications
14. number_outpatient
15. number_emergency
16. number_inpatient
17. diag_1
18. diag_2
19. diag_3
20. number_diagnoses
21. max_glu_serum
22. A1Cresult
23. metformin
24. repaglinide
25. nateglinide
26. chlorpropamide
27. glimepiride
28. acetohexamide
29. glipizide
30. glyburide
31. tolbutamide
32. pioglitazone
33. rosiglitazone
34. acarbose
35. miglitol
36. troglitazone
37. tolazamide
38. examide
39. citoglipton
40. insulin
41. glyburide-metformin
42. glipizide-metformin
43. glimepiride-pioglitazone
44. metformin-rosiglitazone
45. metformin-pioglitazone
46. change
47. diabetesMed


In [26]:
# First 5 records

display(X.head())

,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,...,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed
0,Caucasian,Female,[0-10),NaN,6,25,1,1,NaN,Pediatrics-Endocrinology,...,No,No,No,No,No,No,No,No,No,No
1,Caucasian,Female,[10-20),NaN,1,1,7,3,NaN,NaN,...,No,No,Up,No,No,No,No,No,Ch,Yes
2,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,NaN,NaN,...,No,No,No,No,No,No,No,No,No,Yes
3,Caucasian,Male,[30-40),NaN,1,1,7,2,NaN,NaN,...,No,No,Up,No,No,No,No,No,Ch,Yes
4,Caucasian,Male,[40-50),NaN,1,1,7,1,NaN,NaN,...,No,No,Steady,No,No,No,No,No,Ch,Yes


In [27]:
# Target values

display(y.head(10))

,readmitted
0,NO
1,>30
2,NO
3,NO
4,NO
5,>30
6,NO
7,>30
8,NO
9,NO


In [28]:
print("Readmission target distribution:")
print(y.iloc[:, 0].value_counts())

print("\nReadmission percentages:")
print(
    y.iloc[:, 0]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Readmission target distribution:
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

Readmission percentages:
readmitted
NO     53.91
>30    34.93
<30    11.16
Name: proportion, dtype: float64


In [29]:
# Missing values

missing = X.isnull().sum()

missing_df = (
    pd.DataFrame({
        "Missing_Count": missing,
        "Missing_Percentage": (missing / len(X) * 100).round(2)
    })
    .sort_values("Missing_Percentage", ascending=False)
)

display(missing_df)

,Missing_Count,Missing_Percentage
weight,98569,96.86
max_glu_serum,96420,94.75
A1Cresult,84748,83.28
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02
troglitazone,0,0.00


In [30]:
# Data types of all columns

dtype_df = pd.DataFrame({
    "Column": X.columns,
    "Data_Type": X.dtypes.astype(str).values
})

display(dtype_df)

,Column,Data_Type
0,race,str
1,gender,str
2,age,str
3,weight,str
4,admission_type_id,int64
5,discharge_disposition_id,int64
6,admission_source_id,int64
7,time_in_hospital,int64
8,payer_code,str
9,medical_specialty,str


In [31]:
# Duplicate rows

duplicate_count = X.duplicated().sum()

print("Duplicate rows:", duplicate_count)
print("Duplicate percentage:", round(duplicate_count / len(X) * 100, 2), "%")

Duplicate rows: 0
Duplicate percentage: 0.0 %


In [32]:
# ============================================================
# RAW DATA QUALITY SUMMARY
# ============================================================

print("=" * 60)
print("DATASET SHAPE")
print("=" * 60)
print("Rows:", X.shape[0])
print("Columns:", X.shape[1])

print("\n" + "=" * 60)
print("TARGET DISTRIBUTION")
print("=" * 60)

target = y.iloc[:, 0]

target_summary = pd.DataFrame({
    "Count": target.value_counts(),
    "Percentage": target.value_counts(normalize=True).mul(100).round(2)
})

display(target_summary)

print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

missing_summary = pd.DataFrame({
    "Missing_Count": X.isna().sum(),
    "Missing_Percentage": (
        X.isna().sum() / len(X) * 100
    ).round(2)
})

missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values(
    "Missing_Percentage",
    ascending=False
)

display(missing_summary)

print("\n" + "=" * 60)
print("DATA TYPES")
print("=" * 60)

display(
    pd.DataFrame({
        "Column": X.columns,
        "Data_Type": X.dtypes.astype(str).values
    })
)

print("\n" + "=" * 60)
print("DUPLICATES")
print("=" * 60)

print("Duplicate rows:", X.duplicated().sum())

DATASET SHAPE
Rows: 101766
Columns: 47

TARGET DISTRIBUTION


,Count,Percentage
readmitted,,
NO,54864,53.91
>30,35545,34.93
<30,11357,11.16



MISSING VALUES


,Missing_Count,Missing_Percentage
weight,98569,96.86
max_glu_serum,96420,94.75
A1Cresult,84748,83.28
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02



DATA TYPES


,Column,Data_Type
0,race,str
1,gender,str
2,age,str
3,weight,str
4,admission_type_id,int64
5,discharge_disposition_id,int64
6,admission_source_id,int64
7,time_in_hospital,int64
8,payer_code,str
9,medical_specialty,str



DUPLICATES
Duplicate rows: 0


In [33]:
# Check how missing values are represented

for col in ["weight", "max_glu_serum", "A1Cresult", "medical_specialty", "payer_code"]:
    print(f"\n{col}")
    print(X[col].value_counts(dropna=False).head(10))


weight
weight
NaN          98569
[75-100)      1336
[50-75)        897
[100-125)      625
[125-150)      145
[25-50)         97
[0-25)          48
[150-175)       35
[175-200)       11
>200             3
Name: count, dtype: int64

max_glu_serum
max_glu_serum
NaN     96420
Norm     2597
>200     1485
>300     1264
Name: count, dtype: int64

A1Cresult
A1Cresult
NaN     84748
>8       8216
Norm     4990
>7       3812
Name: count, dtype: int64

medical_specialty
medical_specialty
NaN                           49949
InternalMedicine              14635
Emergency/Trauma               7565
Family/GeneralPractice         7440
Cardiology                     5352
Surgery-General                3099
Nephrology                     1613
Orthopedics                    1400
Orthopedics-Reconstructive     1233
Radiologist                    1140
Name: count, dtype: int64

payer_code
payer_code
NaN    40256
MC     32439
HM      6274
SP      5007
BC      4655
MD      3532
CP      2533
UN      2448
CM   

In [34]:
# Detailed inspection of important categorical columns

columns_to_check = [
    "A1Cresult",
    "medical_specialty",
    "payer_code",
    "race",
    "diag_1",
    "diag_2",
    "diag_3"
]

for col in columns_to_check:
    print("\n" + "=" * 60)
    print(col)
    print("=" * 60)
    print(X[col].value_counts(dropna=False).head(15))


A1Cresult
A1Cresult
NaN     84748
>8       8216
Norm     4990
>7       3812
Name: count, dtype: int64

medical_specialty
medical_specialty
NaN                                49949
InternalMedicine                   14635
Emergency/Trauma                    7565
Family/GeneralPractice              7440
Cardiology                          5352
Surgery-General                     3099
Nephrology                          1613
Orthopedics                         1400
Orthopedics-Reconstructive          1233
Radiologist                         1140
Pulmonology                          871
Psychiatry                           854
Urology                              685
ObstetricsandGynecology              671
Surgery-Cardiovascular/Thoracic      652
Name: count, dtype: int64

payer_code
payer_code
NaN    40256
MC     32439
HM      6274
SP      5007
BC      4655
MD      3532
CP      2533
UN      2448
CM      1937
OG      1033
PO       592
DM       549
CH       146
WC       135
OT        95
N

In [35]:
# ============================================================
# STEP 7B — DATA CLEANING
# ============================================================

# Make a copy of the raw features
X_clean = X.copy()

print("Original shape:", X_clean.shape)


# ------------------------------------------------------------
# 1. DROP EXTREMELY SPARSE FEATURES
# ------------------------------------------------------------

# More than 90% values are missing.
# These features are too sparse for reliable modelling.

columns_to_drop = [
    "weight",
    "max_glu_serum"
]

X_clean = X_clean.drop(columns=columns_to_drop)


# ------------------------------------------------------------
# 2. HANDLE MISSING CATEGORICAL VALUES
# ------------------------------------------------------------

# A1Cresult:
# Missing means the test/result was not recorded.
X_clean["A1Cresult"] = X_clean["A1Cresult"].fillna("Not_Measured")

# Medical specialty:
# Missing means specialty information was not available.
X_clean["medical_specialty"] = X_clean["medical_specialty"].fillna("Unknown")

# Payer code:
# Missing information will be treated as Unknown for now.
X_clean["payer_code"] = X_clean["payer_code"].fillna("Unknown")

# Race:
X_clean["race"] = X_clean["race"].fillna("Unknown")

# Diagnosis columns:
X_clean["diag_1"] = X_clean["diag_1"].fillna("Unknown")
X_clean["diag_2"] = X_clean["diag_2"].fillna("Unknown")
X_clean["diag_3"] = X_clean["diag_3"].fillna("Unknown")


# ------------------------------------------------------------
# 3. CLEAN AGE
# ------------------------------------------------------------

# Age is represented as ranges such as [50-60), [60-70), etc.
# We will keep the original category for now.
X_clean["age"] = X_clean["age"].fillna("Unknown")


# ------------------------------------------------------------
# 4. HANDLE GENDER
# ------------------------------------------------------------

X_clean["gender"] = X_clean["gender"].fillna("Unknown")


# ------------------------------------------------------------
# 5. CHECK REMAINING MISSING VALUES
# ------------------------------------------------------------

remaining_missing = X_clean.isna().sum()

remaining_missing = remaining_missing[
    remaining_missing > 0
].sort_values(ascending=False)


print("\nRemaining missing values:")
display(remaining_missing)


# ------------------------------------------------------------
# 6. FINAL SHAPE
# ------------------------------------------------------------

print("\nCleaned shape:", X_clean.shape)

print("\nCleaning completed successfully!")

Original shape: (101766, 47)

Remaining missing values:


Series([], dtype: int64)


Cleaned shape: (101766, 45)

Cleaning completed successfully!


In [36]:
# ============================================================
# STEP 7C — CREATE BINARY TARGET
# ============================================================

target = y.iloc[:, 0].copy()

# 1 = readmitted within 30 days
# 0 = not readmitted within 30 days

y_clean = (target == "<30").astype(int)

print("Target created successfully!\n")

print("Target distribution:")
display(
    y_clean.value_counts().rename(
        index={
            0: "Not readmitted within 30 days",
            1: "Readmitted within 30 days"
        }
    )
)

print("\nTarget percentage:")
display(
    y_clean.value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename(
        index={
            0: "Not readmitted within 30 days",
            1: "Readmitted within 30 days"
        }
    )
)

Target created successfully!

Target distribution:


readmitted
Not readmitted within 30 days    90409
Readmitted within 30 days        11357
Name: count, dtype: int64


Target percentage:


readmitted
Not readmitted within 30 days    88.84
Readmitted within 30 days        11.16
Name: proportion, dtype: float64

In [37]:
# ============================================================
# FINAL CLEAN DATA QUALITY CHECK
# ============================================================

print("X_clean shape:", X_clean.shape)
print("y_clean shape:", y_clean.shape)

print("\nMissing values:", X_clean.isna().sum().sum())
print("Duplicate rows:", X_clean.duplicated().sum())

print("\nTarget values:")
print(y_clean.value_counts())

print("\nData types:")
print(X_clean.dtypes.value_counts())

X_clean shape: (101766, 45)
y_clean shape: (101766,)

Missing values: 0
Duplicate rows: 0

Target values:
readmitted
0    90409
1    11357
Name: count, dtype: int64

Data types:
str      34
int64    11
Name: count, dtype: int64


In [38]:
# ============================================================
# SAVE CLEANED DATA FOR NEXT PROJECT STAGES
# ============================================================

# Add target to cleaned dataset
cleaned_data = X_clean.copy()
cleaned_data["readmitted_30"] = y_clean

# Save cleaned dataset
cleaned_data.to_csv(
    "../data/cleaned_healthcare_data.csv",
    index=False
)

print("Cleaned dataset saved successfully!")
print("Shape:", cleaned_data.shape)
print("File: data/cleaned_healthcare_data.csv")

Cleaned dataset saved successfully!
Shape: (101766, 46)
File: data/cleaned_healthcare_data.csv
